In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [2]:
## Load the pickle file
## Load the trained model, Scaler pickle file and One hot encoder pickle file

model = load_model('model.h5') # loads a saved Keras/TensorFlow deep learning model—including its architecture, trained weights, and compilation/optimizer state—from an HDF5 (.h5) file into the variable

## Load the encoder and scaler

with open('ohe_geo.pkl', 'rb') as file:
    ohe_geo = pickle.load(file)

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)



In [3]:
# Example Input Data must be having the columns that we used to train the model x_train

input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [4]:
## Converting the Geography in similar to trained model data so trained model can understand it
geo_encoded = ohe_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=  ohe_geo.get_feature_names_out(['Geography']))
geo_encoded_df

C:\Users\rohit\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [5]:
## Converting the Input data to dataframe for giving input to trained model
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [6]:
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
# transform() only applies previously learned parameters to modify data
# fit_transform() learns data parameters and applies the transformation in one step
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [7]:
## Concat the geographical encoded values in input_df and remove the existing Geography column
input_df = pd.concat([input_df.drop('Geography', axis= 1), geo_encoded_df], axis = 1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [8]:
## scale input data in machine learning so that features on different numerical ranges contribute fairly and models train faster and more accurately
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [9]:
prediction = model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step


array([[0.05269438]], dtype=float32)

In [10]:
prediction_probab = prediction[0][0]
prediction_probab

np.float32(0.052694377)